# Khối C — fine-tune VietOCR `vgg_transformer` trên crops VAIPE (C3, Kaggle T4)

Plan §5.3. Bật **GPU T4** + **Internet**, add dataset VAIPE. ~20k iter batch 32 ≈ 4–6h.
Trễ: giảm `ITERS` về 10000 (plan §8.4). Tải `vietocr_finetuned.pth` về `code/weights/`.

In [ ]:
!pip install -q vietocr

In [ ]:
import json, os, glob, shutil
from pathlib import Path

# Đổi nếu dataset Kaggle đặt tên khác. Cần cấu trúc <root>/{train,valid,test}/{images,labels}
CANDIDATES = glob.glob("/kaggle/input/*/1_vaipe") + glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")
VAIPE = next(Path(c) for c in CANDIDATES if (Path(c) / "train" / "labels").exists())
WORK = Path("/kaggle/working")
LABELS = ["drugname", "quantity", "usage", "diagnose", "date", "other"]
print("VAIPE =", VAIPE, {s: len(list((VAIPE / s / "images").glob("*.png"))) for s in ["train", "valid", "test"]})

ITERS = 20000
BATCH = 32

In [ ]:
# Cắt crops theo box chuẩn, đệm 2 px, KHÔNG phóng (VietOCR tự resize chiều cao 32).
# Train trên train, valid để theo dõi (plan ghi train+valid; giữ valid riêng để chọn checkpoint). Test chỉ đo E-C1 ở local.
import cv2
C = WORK / "crops"
def make(split):
    (C / split).mkdir(parents=True, exist_ok=True)
    rows = []
    for img_path in sorted((VAIPE / split / "images").glob("*.png")):
        img = cv2.imread(str(img_path)); h, w = img.shape[:2]
        lbl = VAIPE / split / "labels" / f"{img_path.stem}.json"
        for r in (json.loads(lbl.read_text(encoding="utf-8")) if lbl.exists() else []):
            text = (r.get("text") or "").strip()
            x0, y0, x1, y1 = r["box"]
            if not text or x1 <= x0 or y1 <= y0: continue
            crop = img[max(0, y0-2):min(h, y1+2), max(0, x0-2):min(w, x1+2)]
            if crop.size == 0: continue
            name = f"{split}/{img_path.stem}_{r['id']}.png"
            cv2.imwrite(str(C / name), crop)
            rows.append(f"{name}\t{text}")
    (C / f"{split}_label.txt").write_text("\n".join(rows), encoding="utf-8")
    return len(rows)
print({s: make(s) for s in ["train", "valid"]})

In [ ]:
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer
cfg = Cfg.load_config_from_name("vgg_transformer")   # trọng số pretrained tải tự động
cfg["dataset"].update({"name": "vaipe", "data_root": str(C), "train_annotation": "train_label.txt",
                       "valid_annotation": "valid_label.txt"})
cfg["trainer"].update({"iters": ITERS, "batch_size": BATCH, "print_every": 200, "valid_every": 2000,
                       "checkpoint": str(WORK / "vietocr_ckpt.pth"), "export": str(WORK / "vietocr_finetuned.pth"),
                       "metrics": 2000})
cfg["dataloader"].update({"num_workers": 2})
cfg["optimizer"]["max_lr"] = 3e-4    # nhỏ hơn mặc định vì fine-tune từ pretrained
cfg["device"] = "cuda:0"
trainer = Trainer(cfg, pretrained=True)
trainer.train()

In [ ]:
# Độ chính xác trên valid (theo dõi, không phải số liệu báo cáo — E-C1 đo trên test ở local)
print(trainer.precision())
cfg.save(str(WORK / "vietocr_finetuned_config.yml"))
print("→ tải /kaggle/working/vietocr_finetuned.pth về code/weights/")